In [1]:
!pip install gradio

In [2]:
import random

question_bank = {
    "easy": [
        {"q": "What does HTML stand for?", "options": ["Hyper Text Markup Language", "High Tech Modern Language", "Hyperlink and Text Markup Language", "Home Tool Markup Language"], "answer": 0},
        {"q": "Which of these is a programming language?", "options": ["Python", "Photoshop", "Excel", "Chrome"], "answer": 0},
        {"q": "What does CPU stand for?", "options": ["Central Process Unit", "Central Processing Unit", "Computer Personal Unit", "Central Processor Utility"], "answer": 1},
    ],
    "medium": [
        {"q": "Which data structure uses LIFO order?", "options": ["Queue", "Stack", "Array", "Linked List"], "answer": 1},
        {"q": "What is the time complexity of binary search?", "options": ["O(n)", "O(n^2)", "O(log n)", "O(1)"], "answer": 2},
        {"q": "Which HTTP method is typically used to update a resource?", "options": ["GET", "PUT", "DELETE", "TRACE"], "answer": 1},
    ],
    "hard": [
        {"q": "What problem does the CAP theorem describe?", "options": ["Consistency, Availability, Partition tolerance tradeoffs", "Compression, Accuracy, Performance tradeoffs", "Caching, API, Protocol design", "Concurrency, Atomicity, Persistence"], "answer": 0},
        {"q": "Which sorting algorithm has the best average time complexity?", "options": ["Bubble Sort", "Quick Sort", "Selection Sort", "Insertion Sort"], "answer": 1},
        {"q": "In machine learning, what does overfitting mean?", "options": ["Model performs well on new data only", "Model is too simple", "Model memorizes training data and fails to generalize", "Model trains too fast"], "answer": 2},
    ],
}

DIFFICULTY_ORDER = ["easy", "medium", "hard"]

def pick_question(difficulty, asked):
    pool = [i for i in range(len(question_bank[difficulty])) if (difficulty, i) not in asked]
    if not pool:
        asked = {a for a in asked if a[0] != difficulty}
        pool = list(range(len(question_bank[difficulty])))
    idx = random.choice(pool)
    asked.add((difficulty, idx))
    return idx, asked

In [3]:
def init_state():
    state = {"difficulty": "medium", "streak_correct": 0, "streak_incorrect": 0,
             "asked": set(), "current_idx": None, "score": 0, "total": 0}
    idx, asked = pick_question(state["difficulty"], state["asked"])
    state["current_idx"] = idx
    state["asked"] = asked
    q = question_bank[state["difficulty"]][idx]
    status = f"Difficulty: {state['difficulty'].title()} | Score: {state['score']}/{state['total']}"
    return state, q["q"], gr.update(choices=q["options"], value=None), status

def submit_answer(selected, state):
    if selected is None:
        q = question_bank[state["difficulty"]][state["current_idx"]]
        status = f"Difficulty: {state['difficulty'].title()} | Score: {state['score']}/{state['total']}"
        return state, "Please select an answer.", q["q"], gr.update(choices=q["options"], value=None), status

    q = question_bank[state["difficulty"]][state["current_idx"]]
    is_correct = (q["options"].index(selected) == q["answer"])
    state["total"] += 1

    if is_correct:
        state["score"] += 1
        state["streak_correct"] += 1
        state["streak_incorrect"] = 0
        feedback = "Correct!"
    else:
        state["streak_incorrect"] += 1
        state["streak_correct"] = 0
        feedback = f"Incorrect. The correct answer was: {q['options'][q['answer']]}"

    # This is the adaptive step: difficulty shifts based on recent performance
    level_idx = DIFFICULTY_ORDER.index(state["difficulty"])
    if state["streak_correct"] >= 2 and level_idx < len(DIFFICULTY_ORDER) - 1:
        state["difficulty"] = DIFFICULTY_ORDER[level_idx + 1]
        state["streak_correct"] = 0
        feedback += " Moving up to a harder difficulty."
    elif state["streak_incorrect"] >= 1 and level_idx > 0:
        state["difficulty"] = DIFFICULTY_ORDER[level_idx - 1]
        state["streak_incorrect"] = 0
        feedback += " Moving down to an easier difficulty."

    idx, asked = pick_question(state["difficulty"], state["asked"])
    state["current_idx"] = idx
    state["asked"] = asked
    next_q = question_bank[state["difficulty"]][idx]
    status = f"Difficulty: {state['difficulty'].title()} | Score: {state['score']}/{state['total']}"
    return state, feedback, next_q["q"], gr.update(choices=next_q["options"], value=None), status

In [4]:
import gradio as gr

with gr.Blocks(title="Adaptive Difficulty Quiz") as demo:
    gr.Markdown("# Adaptive Difficulty Quiz\nAnswer questions and the difficulty adapts to your performance in real time.")
    status_box = gr.Textbox(label="Status", interactive=False)
    question_box = gr.Textbox(label="Question", interactive=False)
    options_radio = gr.Radio(choices=[], label="Choose an answer")
    feedback_box = gr.Textbox(label="Feedback", interactive=False)
    submit_btn = gr.Button("Submit Answer")
    state = gr.State()

    demo.load(fn=init_state, outputs=[state, question_box, options_radio, status_box])
    submit_btn.click(fn=submit_answer, inputs=[options_radio, state],
                      outputs=[state, feedback_box, question_box, options_radio, status_box])

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
